In [ ]:
import random
from pathlib import Path

import pyarrow.parquet as pq
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from tokenizers.normalizers import NFC
from tqdm.auto import tqdm
from transformers import PreTrainedTokenizerFast

# nbconvert runs from notebooks/, so find the repo root
repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())

corpus_dir = repo_root / "data/corpus/habsburg_clean"
tokenizer_dir = repo_root / "data/tokenizer/habsburg-48k"

vocab_size = 48000
sample_fraction = 0.03  # ~3B of 103B characters
seed = 42
batch_size = 1000


In [ ]:
tokenizer = Tokenizer(models.BPE())
tokenizer.normalizer = NFC()
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=["<|endoftext|>", "<|padding|>"],
    min_frequency=2,
)

corpus_files = sorted(corpus_dir.glob("*.parquet"))
total_documents = sum(pq.ParquetFile(f).metadata.num_rows for f in corpus_files)
print(f"Found {len(corpus_files)} year files, {total_documents:,} documents.")


In [ ]:
def document_iterator(files, sample_fraction, seed, batch_size):
    rng = random.Random(seed)
    batch = []
    kept = 0
    progress = tqdm(files, desc="Reading years", unit="file")
    for parquet_path in progress:
        for record_batch in pq.ParquetFile(parquet_path).iter_batches(
            batch_size=256, columns=["text"]
        ):
            for text in record_batch.column("text").to_pylist():
                if rng.random() >= sample_fraction:
                    continue
                batch.append(text)
                kept += 1
                if len(batch) >= batch_size:
                    progress.set_postfix(documents=f"{kept:,}")
                    yield batch
                    batch = []
    if batch:
        yield batch


tokenizer.train_from_iterator(
    document_iterator(corpus_files, sample_fraction, seed, batch_size),
    trainer=trainer,
)
print(f"Trained vocab size: {tokenizer.get_vocab_size()}")


In [ ]:
tokenizer_dir.mkdir(parents=True, exist_ok=True)

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    eos_token="<|endoftext|>",
    pad_token="<|padding|>",
)
hf_tokenizer.save_pretrained(tokenizer_dir)
print(f"Saved to {tokenizer_dir}")


In [ ]:
samples = [
    "Se. k. k. Majestät haben der Deputation aus der Lombardey eine feyerliche Audienz ertheilet.",
    "Die Donaudampfschiffahrtsgesellschaft eröffnete die Linie nach Preßburg.",
    "1840 1850 1860 1870 1880 1890 1900 1910",
]

for sample in samples:
    encoded = hf_tokenizer.encode(sample)
    print(f"Text:    {sample}")
    print(f"Count:   {len(encoded)} tokens")
    print(f"Pieces:  {hf_tokenizer.convert_ids_to_tokens(encoded)}")
    print(f"Decoded: {hf_tokenizer.decode(encoded)}")
    print("-" * 40)
